# Datetime Handling

Working with dates and times is essential when analyzing time series data from building sensors and energy meters. pandas provides powerful tools for parsing, manipulating, and indexing datetime data.

In [ ]:
import pandas as pd

## Converting Strings to Datetime

When loading CSV files, date and time columns are typically read as strings (`object` dtype). Use `pd.to_datetime()` to convert them.

In [ ]:
url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatElectricity.csv"

df = pd.read_csv(url, sep=";")
df.head()

In [ ]:
df.dtypes

```{note}
The `time` column has dtype `object` — it is still a string. We need to convert it to `datetime64`.
```

In [ ]:
df["time"] = pd.to_datetime(df["time"])
df.dtypes

## Format Strings

If the date format is non-standard, you can specify a format string to help pandas parse it correctly.

| Code | Meaning | Example |
|------|---------|--------|
| `%Y` | 4-digit year | 2023 |
| `%m` | Month (zero-padded) | 01, 12 |
| `%d` | Day (zero-padded) | 01, 31 |
| `%H` | Hour (24h) | 00, 23 |
| `%M` | Minute | 00, 59 |
| `%S` | Second | 00, 59 |

In [ ]:
# Example with explicit format
dates = pd.Series(["03.10.2018 00:00", "03.10.2018 01:00", "03.10.2018 02:00"])
pd.to_datetime(dates, format="%d.%m.%Y %H:%M")

```{tip}
Specifying the `format` parameter explicitly is faster than letting pandas infer it, especially for large data sets.
```

## Setting a Datetime Index

Setting the datetime column as the DataFrame index enables powerful time-based slicing and resampling.

In [ ]:
df = df.set_index("time")
df.head()

In [ ]:
df.index

In [ ]:
# Slice by date range
df.loc["2019-01-01":"2019-01-03"]

## The `.dt` Accessor

The `.dt` accessor provides convenient properties for extracting components from datetime columns.

```{note}
The `.dt` accessor works on **columns** (Series), not on a DatetimeIndex. If the datetime is the index, access the properties directly (e.g., `df.index.year`).
```

In [ ]:
# Reset index to use .dt accessor on the column
df_reset = df.reset_index()
df_reset.head()

In [ ]:
# Extract year
df_reset["time"].dt.year.head()

In [ ]:
# Extract month
df_reset["time"].dt.month.head()

In [ ]:
# Extract day
df_reset["time"].dt.day.head()

In [ ]:
# Extract hour
df_reset["time"].dt.hour.head()

In [ ]:
# Day of the week (Monday=0, Sunday=6)
df_reset["time"].dt.dayofweek.head()

In [ ]:
# Custom string formatting
df_reset["time"].dt.strftime("%A, %d %B %Y").head()

## Adding Derived Columns

The `.dt` accessor is useful for creating new columns that group data by time period.

In [ ]:
df_reset["year"] = df_reset["time"].dt.year
df_reset["month"] = df_reset["time"].dt.month
df_reset["hour"] = df_reset["time"].dt.hour
df_reset["day_of_week"] = df_reset["time"].dt.dayofweek

df_reset.head()

## Time Zones

By default, datetime objects in pandas are timezone-naive. You can localize them to a specific timezone and convert between timezones.

In [ ]:
# Create a small example
dates = pd.date_range("2023-06-15 12:00", periods=4, freq="h")
s = pd.Series([22.1, 22.5, 23.0, 22.8], index=dates)
s

In [ ]:
# Localize to a timezone (assign a timezone to naive datetimes)
s_zurich = s.tz_localize("Europe/Zurich")
s_zurich

In [ ]:
# Convert to another timezone
s_utc = s_zurich.tz_convert("UTC")
s_utc

## Creating Date Ranges

`pd.date_range()` creates a sequence of evenly spaced datetime values. This is useful for creating reference time axes or filling gaps.

In [ ]:
# Hourly timestamps for one week
date_range = pd.date_range(start="2023-01-01", end="2023-01-07", freq="h")
print(f"Number of timestamps: {len(date_range)}")
date_range[:10]

In [ ]:
# Daily timestamps for one year
daily_range = pd.date_range(start="2023-01-01", periods=365, freq="D")
daily_range[:10]

In [ ]:
# 15-minute intervals
quarter_hourly = pd.date_range(start="2023-01-01", periods=96, freq="15min")
quarter_hourly[:10]

```{tip}
Common frequency strings: `"h"` (hourly), `"D"` (daily), `"W"` (weekly), `"MS"` (month start), `"15min"` (15 minutes), `"min"` (1 minute).
```

## Practical Example

Let's put it all together with the electricity data set.

In [ ]:
url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatElectricity.csv"

df = pd.read_csv(url, sep=";")

# Convert time column
df["time"] = pd.to_datetime(df["time"])

# Set as index
df = df.set_index("time")

df.head()

In [ ]:
# Check the time range of the data
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")
print(f"Duration: {df.index.max() - df.index.min()}")

In [ ]:
# Select data for a specific month
jan_2019 = df.loc["2019-01"]
jan_2019.head()

In [ ]:
# Resample to daily values
df_daily = df.resample("D").mean()
df_daily.head()